In [2]:
from typing_extensions import TypedDict, Annotated
from typing import Any
from langchain_core.messages import HumanMessage, AnyMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from openai import AzureOpenAI
from dotenv import load_dotenv
import os
import json
# Load environment variables
load_dotenv()

# Configuration
end_point = os.getenv("AZURE_OPENAI_GPT_4O_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_GPT_4O_API_KEY")
api_version = os.getenv("AZURE_OPENAI_GPT_4O_API_VERSION")
deployment = os.getenv("AZURE_OPENAI_GPT_4O_DEPLOYMENT_NAME").strip()

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=end_point,
    api_key=api_key
)
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def get_person_details(person_name: str) -> str:
    """Get person details."""
    return f"{person_name} is a DevOps Engineer."

# Tool: Location Information
def get_person_location(person_name: str) -> str:
    """Get person location."""
    return f"{person_name} lives in Bangalore."

# System message
sys_msg = SystemMessage(
    content="You are a helpful assistant that provides accurate responses based on the given tools."
)
def assistant(state: MessagesState):
    messages = [{"role": "system", "content": sys_msg.content}]
    for msg in state["messages"]:
        messages.append({"role": "user", "content": msg.content})

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "get_person_details",
                    "description": "Retrieve details of a person",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "person_name": {"type": "string"}
                        },
                        "required": ["person_name"]
                    }
                }
            },
            {
                "type": "function",
                "function": {
                    "name": "get_person_location",
                    "description": "Retrieve location of a person",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "person_name": {"type": "string"}
                        },
                        "required": ["person_name"]
                    }
                }
            }
        ],
        tool_choice="auto"
    )

    choice = response.choices[0]

    if choice.finish_reason == "tool_calls":
        tool_call = choice.message.tool_calls[0]
        args = json.loads(tool_call.function.arguments)
        if tool_call.function.name == "get_person_details":
            result = get_person_details(**args)
        elif tool_call.function.name == "get_person_location":
            result = get_person_location(**args)
        return {"messages": [HumanMessage(content=result)]}

    reply = choice.message.content or "Sorry, I don't have a response for that."
    return {"messages": [HumanMessage(content=reply)]}
builder = StateGraph(MessagesState)
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode([get_person_details, get_person_location]))

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", tools_condition)
builder.add_edge("tools", "assistant")

memory = MemorySaver()
react_graph = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "unique_conversation_1"}}
messages = [HumanMessage(content="Who is Alice?")]
result = react_graph.invoke({"messages": messages}, config)
for message in result['messages']:
    message.pretty_print()
    messages = [HumanMessage(content="Where does Alice live?")]
result = react_graph.invoke({"messages": messages}, config)
for message in result['messages']:
    message.pretty_print()

messages = [HumanMessage(content="hi! My name is Khush?")]
result = react_graph.invoke({"messages": messages}, config)
for message in result['messages']:
    message.pretty_print()

messages = [HumanMessage(content="What do you know about Alice?")]
result = react_graph.invoke({"messages": messages}, config)
for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

Who is Alice?
================================ Human Message =================================

Alice is a DevOps Engineer.
================================ Human Message =================================

Who is Alice?
================================ Human Message =================================

Alice is a DevOps Engineer.
================================ Human Message =================================

Where does Alice live?
================================ Human Message =================================

Alice lives in Bangalore.
================================ Human Message =================================

Who is Alice?
================================ Human Message =================================

Alice is a DevOps Engineer.
================================ Human Message =================================

Where does Alice live?
================================ Human Message =================